In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

print(sys.executable)  #

/home/nasim/Desktop/work/rag/sec-10k-rag/.venv/bin/python


In [3]:
from sec_10k_rag.split_text.lib import  load_ndjson, drop_short_sections, chunk_sections 

In [4]:
import polars as pl
from pathlib import Path

In [5]:
PRJ_ROOT_DIR = Path.cwd().parent
MIN_CHARS = 150
TARGET_CHARS = 500
META_COLUMNS = [
    "section_id",
    "ticker",
    "cik",
    "fiscal_year",
    "item",
    "title",
    "report_date",
    "source_url",
]
DATA_DIR = Path("data/processed")

In [6]:
df = load_ndjson(PRJ_ROOT_DIR / DATA_DIR / "sections.jsonl")

In [7]:
with pl.Config(tbl_cols=-1, fmt_str_lengths=TARGET_CHARS, tbl_rows=-1):
    print(df.filter(pl.col("n_chars") > 80).filter(pl.col("n_chars") < 155).sort(pl.col("n_chars")).select(pl.exclude("text")))

shape: (15, 11)
┌─────────┬────────┬─────────┬────────┬────────┬────────┬────────┬────────┬──────┬────────┬────────┐
│ section ┆ ticker ┆ cik     ┆ fiscal ┆ report ┆ access ┆ source ┆ source ┆ item ┆ title  ┆ n_char │
│ _id     ┆ ---    ┆ ---     ┆ _year  ┆ _date  ┆ ion    ┆ _forma ┆ _url   ┆ ---  ┆ ---    ┆ s      │
│ ---     ┆ str    ┆ str     ┆ ---    ┆ ---    ┆ ---    ┆ t      ┆ ---    ┆ str  ┆ str    ┆ ---    │
│ str     ┆        ┆         ┆ i64    ┆ str    ┆ str    ┆ ---    ┆ str    ┆      ┆        ┆ i64    │
│         ┆        ┆         ┆        ┆        ┆        ┆ str    ┆        ┆      ┆        ┆        │
╞═════════╪════════╪═════════╪════════╪════════╪════════╪════════╪════════╪══════╪════════╪════════╡
│ NVDA-20 ┆ NVDA   ┆ 0001045 ┆ 2026   ┆ 2026-0 ┆ 000104 ┆ html   ┆ https: ┆ 9    ┆ Change ┆ 98     │
│ 26-item ┆        ┆ 810     ┆        ┆ 1-25   ┆ 5810-2 ┆        ┆ //www. ┆      ┆ s in   ┆        │
│ 9       ┆        ┆         ┆        ┆        ┆ 6-0000 ┆        ┆ sec.go ┆

In [8]:
r1 = drop_short_sections(df, MIN_CHARS)

In [9]:
with pl.Config(tbl_cols=-1, tbl_width_chars=-1, fmt_str_lengths=10): print(r1)

shape: (168, 12)
┌─────────────┬────────┬────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬──────┬─────────────┬─────────┬─────────────┐
│ section_id  ┆ ticker ┆ cik        ┆ fiscal_yea… ┆ report_dat… ┆ accession   ┆ source_for… ┆ source_url  ┆ item ┆ title       ┆ n_chars ┆ text        │
│ ---         ┆ ---    ┆ ---        ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---  ┆ ---         ┆ ---     ┆ ---         │
│ str         ┆ str    ┆ str        ┆ i64         ┆ str         ┆ str         ┆ str         ┆ str         ┆ str  ┆ str         ┆ i64     ┆ str         │
╞═════════════╪════════╪════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪══════╪═════════════╪═════════╪═════════════╡
│ MSFT-2026-… ┆ MSFT   ┆ 0000789019 ┆ 2026        ┆ 2026-06-30  ┆ 0001193125… ┆ html        ┆ https://ww… ┆ 1    ┆ Business    ┆ 42878   ┆ Item 1      │
│             ┆        ┆            ┆             ┆             ┆

In [10]:
r2 = chunk_sections(r1, target_chars=TARGET_CHARS, meta_columns=META_COLUMNS)

In [11]:
with pl.Config(fmt_str_lengths=20, tbl_rows=-1, tbl_cols=-1, tbl_width_chars=-1): 
    display(r2.head(10))

chunk_index,text,section_id,ticker,cik,fiscal_year,item,title,report_date,source_url
i64,str,str,str,str,i64,str,str,str,str
0,"""Item 1 Note About Fo…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
1,"""PART I ITEM 1. BUSIN…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
2,"""We develop and suppo…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
3,"""We believe AI should…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
4,"""We offer an array of…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
5,"""Digital transformati…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
6,"""We prioritize securi…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
7,"""•Build the intellige…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"
8,"""•Tackling security f…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/…"


In [12]:
display(r2.head(10))

chunk_index,text,section_id,ticker,cik,fiscal_year,item,title,report_date,source_url
i64,str,str,str,str,i64,str,str,str,str
0,"""Item 1 Note About Forward-Look…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
1,"""PART I ITEM 1. BUSINESS GENERA…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
2,"""We develop and support a broad…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
3,"""We believe AI should be as emp…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
4,"""We offer an array of services,…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
5,"""Digital transformation and ado…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
6,"""We prioritize security above a…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
7,"""•Build the intelligent cloud a…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"
8,"""•Tackling security from all an…","""MSFT-2026-item1""","""MSFT""","""0000789019""",2026,"""1""","""Business""","""2026-06-30""","""https://www.sec.gov/Archives/e…"


In [17]:
def preview_chunks(chunks: pl.DataFrame, n: int = 150) -> pl.DataFrame:
    is_long = pl.col("text").str.len_chars() > n
    return (
        chunks.sort("section_id", "chunk_index")
        .with_columns(
            pl.when(is_long).then(pl.col("text").str.slice(0, n) + "...").otherwise(pl.col("text")).alias(f"first_{n}"),
            pl.when(is_long).then("..." + pl.col("text").str.slice(-n)).otherwise(pl.col("text")).alias(f"last_{n}"),
        )
        .select("section_id", "ticker", f"first_{n}", f"last_{n}", "text", "chunk_index")
    )


In [18]:
report2 = preview_chunks(r2, 50)

In [19]:
with pl.Config(fmt_str_lengths=60, tbl_rows=20, tbl_cols=-1, tbl_width_chars=-1):
    display(report2.slice(270, 40))

section_id,ticker,first_50,last_50,text,chunk_index
str,str,str,str,str,i64
"""MSFT-2024-item7""","""MSFT""","""LinkedIn revenue growth Revenue from LinkedIn, inc...""","""...s”); and Nuance and GitHub More Personal Computing""","""LinkedIn revenue growth Revenue from LinkedIn, including Tal…",22
"""MSFT-2024-item7""","""MSFT""","""Metrics related to our More Personal Computing seg...""","""... and non-Pro licenses sold through the OEM channel""","""Metrics related to our More Personal Computing segment asses…",23
"""MSFT-2024-item7""","""MSFT""","""Windows Commercial products and cloud services rev...""","""...ird-party disc royalties; and other cloud services""","""Windows Commercial products and cloud services revenue growt…",24
"""MSFT-2024-item7""","""MSFT""","""Search and news advertising revenue (ex TAC) growt...""","""...xcept percentages and per share amounts) 2024 2023""","""Search and news advertising revenue (ex TAC) growth Revenue …",25
"""MSFT-2024-item7""","""MSFT""","""Percentage Change Revenue $ 245,122 $ 211,915 16% ...""","""...ccordance with GAAP to non-GAAP financial results.""","""Percentage Change Revenue $ 245,122 $ 211,915 16% Gross marg…",26
"""MSFT-2024-item7""","""MSFT""","""Fiscal Year 2024 Compared with Fiscal Year 2023 Re...""","""... 17% driven by growth across each of our segments.""","""Fiscal Year 2024 Compared with Fiscal Year 2023 Revenue incr…",27
"""MSFT-2024-item7""","""MSFT""","""•Gross margin percentage increased slightly. Exclu...""","""...cture, offset in part by sales mix shift to Azure.""","""•Gross margin percentage increased slightly. Excluding the i…",28
"""MSFT-2024-item7""","""MSFT""","""Operating expenses increased $4.0 billion or 7% dr...""","""....2 billion, $946 million, and $0.13, respectively.""","""Operating expenses increased $4.0 billion or 7% driven by Ga…",29
"""MSFT-2024-item7""","""MSFT""","""PART II Item 7 SEGMENT RESULTS OF OPERATIONS (In m...""","""...erating Income Productivity and Business Processes""","""PART II Item 7 SEGMENT RESULTS OF OPERATIONS (In millions, e…",30


In [20]:
with pl.Config(fmt_str_lengths=60, tbl_rows=10, tbl_cols=-1, tbl_width_chars=-1):
    display(report2.head(10))

section_id,ticker,first_50,last_50,text,chunk_index
str,str,str,str,str,i64
"""MSFT-2024-item1""","""MSFT""","""Item 1 Note About Forward-Looking Statements This ...""","""...e of new information, future events, or otherwise.""","""Item 1 Note About Forward-Looking Statements This report inc…",0
"""MSFT-2024-item1""","""MSFT""","""PART I ITEM 1. BUSINESS GENERAL Embracing Our Futu...""","""...wth, and impact in every country around the world.""","""PART I ITEM 1. BUSINESS GENERAL Embracing Our Future Microso…",1
"""MSFT-2024-item1""","""MSFT""","""We have entered a new age of AI that will fundamen...""","""...lock for every country, community, and individual.""","""We have entered a new age of AI that will fundamentally tran…",2
"""MSFT-2024-item1""","""MSFT""","""We believe AI should be as empowering across commu...""","""...nd built with safety and security from the outset.""","""We believe AI should be as empowering across communities as …",3
"""MSFT-2024-item1""","""MSFT""","""What We Offer Founded in 1975, we develop and supp...""","""... relevant online advertising to a global audience.""","""What We Offer Founded in 1975, we develop and support softwa…",4
"""MSFT-2024-item1""","""MSFT""","""Our products include operating systems, cross-devi...""","""...ther intelligent devices, and related accessories.""","""Our products include operating systems, cross-device product…",5
"""MSFT-2024-item1""","""MSFT""","""PART I Item 1 The Ambitions That Drive Us To achie...""","""...in the ways the world works, learns, and connects.""","""PART I Item 1 The Ambitions That Drive Us To achieve our vis…",6
"""MSFT-2024-item1""","""MSFT""","""Our growth depends on securely delivering continuo...""","""...ons, knowledge, learning, resources, and insights.""","""Our growth depends on securely delivering continuous innovat…",7
"""MSFT-2024-item1""","""MSFT""","""Together, the Microsoft Cloud, Dynamics 365, Micro...""","""... professionals learn, sell, market, and get hired.""","""Together, the Microsoft Cloud, Dynamics 365, Microsoft Teams…",8
